In [1]:
import torch
import torch.nn as nn
import math

In [2]:
class MultiQueryAttention(nn.Module):
    def __init__(self, d_model=256, n_heads=4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, self.head_dim)
        self.v_proj = nn.Linear(d_model, self.head_dim)
        self.out_proj = nn.Linear(d_model, d_model)

        def forward(self, x, past_cache=None):
        B, S, D = x.shape
        q = self.q_proj(x) 
        k = self.k_proj(x) 
        v = self.v_proj(x) 
        q = q.view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, S, 1, self.head_dim).transpose(1, 2)
        v = v.view(B, S, 1, self.head_dim).transpose(1, 2)

        if past_cache is not None:
            past_k, past_v = past_cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v) 
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        
        return self.out_proj(out), new_cache

# Exercise

In [4]:
d_model = 512
n_heads = 8
head_dim = d_model // n_heads

print("head_dim:", head_dim)

head_dim: 64


In [5]:
past_k = torch.randn(1, 1, 10, head_dim)
past_v = torch.randn(1, 1, 10, head_dim)

print("past_k shape:", past_k.shape)
print("past_v shape:", past_v.shape)

past_k shape: torch.Size([1, 1, 10, 64])
past_v shape: torch.Size([1, 1, 10, 64])


In [6]:
x_new = torch.randn(1, 1, d_model)

print("x_new shape:", x_new.shape)

x_new shape: torch.Size([1, 1, 512])


In [7]:
q_proj = nn.Linear(d_model, d_model)
k_proj = nn.Linear(d_model, head_dim)
v_proj = nn.Linear(d_model, head_dim)

q = q_proj(x_new)
q = q.view(1, 1, n_heads, head_dim).transpose(1, 2)

print("q shape:", q.shape)

q shape: torch.Size([1, 8, 1, 64])


In [8]:
k = k_proj(x_new)
v = v_proj(x_new)

k = k.view(1, 1, 1, head_dim)
v = v.view(1, 1, 1, head_dim)

print("new k shape:", k.shape)
print("new v shape:", v.shape)

new k shape: torch.Size([1, 1, 1, 64])
new v shape: torch.Size([1, 1, 1, 64])


In [9]:
k = torch.cat([past_k, k], dim=2)
v = torch.cat([past_v, v], dim=2)

print("cached k shape:", k.shape)
print("cached v shape:", v.shape)

cached k shape: torch.Size([1, 1, 11, 64])
cached v shape: torch.Size([1, 1, 11, 64])


In [11]:
k = k.expand(-1, n_heads, -1, -1)
v = v.expand(-1, n_heads, -1, -1)

print("expanded k shape:", k.shape)
print("expanded v shape:", v.shape)

expanded k shape: torch.Size([1, 8, 11, 64])
expanded v shape: torch.Size([1, 8, 11, 64])


In [12]:
scores = torch.matmul(q, k.transpose(-2, -1))
print("scores shape:", scores.shape)

assert scores.shape == (1, n_heads, 1, 11)

print("Assertion passed.")

scores shape: torch.Size([1, 8, 1, 11])
Assertion passed.
